In [ ]:
# import libraries

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

In [7]:

# load dataset
data=pd.read_csv("/content/student-por.csv")
print(data.shape)
print(data.columns)

(649, 33)
Index(['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu',
       'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime',
       'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery',
       'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc',
       'Walc', 'health', 'absences', 'G1', 'G2', 'G3'],
      dtype='object')


In [6]:
# Features and Target
y=data['G3']
x=data.drop(['G1','G2','G3'],axis=1) # Remove G1 and G2 to avoid data leakage

In [8]:
# Train/Test Split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [10]:
# Identify Numerical and Categorical Features

numerical_cols=x.select_dtypes(include="number").columns
categorical_cols=x.select_dtypes(include="object").columns


In [11]:

# Data Preprocessing

numeric_transformer=Pipeline([
    ("imputer",SimpleImputer(strategy="median"))
])

categorical_transformer=Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore"))
])
#column Transformer
preprocessor=ColumnTransformer(
    transformers=[
        ("num",numeric_transformer,numerical_cols),
    ("cat",categorical_transformer,categorical_cols)
    ]
)


In [16]:

# Function to Train and Evaluate Models


def evaluate_model(model, model_name):

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", model)
    ])

    pipeline.fit(x_train, y_train)

    predictions = pipeline.predict(x_test)

    mae = mean_absolute_error(y_test, predictions)

    print(f"{model_name}")
    print(f"MAE : {mae:.4f}")
    print("-"*40)

    return mae

In [15]:

# Machine Learning Models


models = {

    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(random_state=42),

    "Random Forest": RandomForestRegressor(random_state=42),

    "XGBoost": XGBRegressor(
        random_state=42
    )

}

In [17]:

# Compare Models


results = {}

for name, model in models.items():

    results[name] = evaluate_model(model, name)

Linear Regression
MAE : 2.1564
----------------------------------------
Decision Tree
MAE : 2.8692
----------------------------------------
Random Forest
MAE : 2.0396
----------------------------------------
XGBoost
MAE : 2.1986
----------------------------------------


In [18]:

# Best Model

best_model = min(results, key=results.get)

print("Best Model :", best_model)

print("Best MAE :", results[best_model])

Best Model : Random Forest
Best MAE : 2.0396153846153844


# New section